# LoRA Fine-Tuning and Neural Transfer

This notebook evaluates LoRA fine-tuning across six neural fine-tuning datasets and eight evaluation benchmarks. The LoRA sweep uses ViT-S at $20$ epochs with ranks $r \in \{8,16,32\}$ and learning rates $\eta \in \{10^{-5},10^{-4},10^{-3}\}$. The analysis:

1. Characterizes the LoRA rank and learning-rate sweep across datasets and benchmarks.
2. Uses one overall LoRA setting for the neural-transfer comparison: $20$ epochs, $r=32$, and $\eta=10^{-4}$, the highest-performing shared setting in the evaluated sweep.
3. Compares this shared LoRA setting with full fine-tuning fixed at $20$ epochs and learning rate $10^{-5}$.
4. Reports dataset-specific LoRA selection only as a descriptive sensitivity analysis in the appendix.

Throughout, all alignment scores are reported as changes relative to the pretrained ViT-S ImageNet baseline.

In [ ]:
from pathlib import Path
import re
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display, Markdown

In [ ]:
repo_dir = Path('../../..')

if str(repo_dir) not in sys.path:
    sys.path.append(str(repo_dir))

from visualization.src.utils import (
    save_figs,
    apply_hiearchical_aggregation,
    BENCHMARK_NAME_MAPPING,
    BENCHMARK_COLORS,
    ARCHITECUTURE_FAMILY_COLORS,
)

In [ ]:
sns.set_theme(style='ticks')
plt.rcParams['figure.dpi'] = 140
plt.rcParams['axes.titleweight'] = 'bold'

artifacts_dir = repo_dir / 'extract_n_eval' / 'artifacts'
save_dir_supp = repo_dir / 'visualization' / 'paper' / 'supp' / 'figures'
save_dir_main = repo_dir / 'visualization' / 'paper' / 'main' / 'figures'

assert artifacts_dir.exists(), f'artifacts directory {artifacts_dir} does not exist!'

In [ ]:
HEATMAP_VAL_MULTIPLIER = 100

METRICS = ['pearsonr_nc', 'rsa_c_test', 'cka_c_test']
AGG_COLS = METRICS + ['task_evaluation_acc']

BENCHMARKS = [
    'bs_fz',
    'bs_mh',
    'tvsd',
    'things_fmri',
    'nsd_func1pt8mm_individualROIs',
    'things_eeg1',
    'things_eeg2',
    'things_meg',
]
BENCHMARKS_WITH_AVERAGE = BENCHMARKS + ['benchmark_average']
BENCHMARK_ORDER = {benchmark: idx for idx, benchmark in enumerate(BENCHMARKS_WITH_AVERAGE)}

FINETUNING_DATASETS = [
    'tvsd',
    'things_fmri',
    'nsd_func1pt8mm_individualROIs',
    'things_eeg1',
    'things_eeg2',
    'things_meg',
]
FINETUNING_DATASET_LABELS = {dataset: BENCHMARK_NAME_MAPPING[dataset] for dataset in FINETUNING_DATASETS}

MODALITY_MAP = {
    'bs_fz': 'EP',
    'bs_mh': 'EP',
    'tvsd': 'EP',
    'things_fmri': 'fMRI',
    'nsd_func1pt8mm_individualROIs': 'fMRI',
    'things_eeg1': 'EEG/MEG',
    'things_eeg2': 'EEG/MEG',
    'things_meg': 'EEG/MEG',
}

BASE_MODEL_ID = 'deit_small_imagenet_full_seed-0'
FULL_FT_EPOCHS = 20
FULL_FT_LR_LABEL = '1e-5'

METHOD_PALETTE = {
    'Full FT': ARCHITECUTURE_FAMILY_COLORS['ViT'],
    'Overall LoRA': '#2A9D8F',
    'Best LoRA (per dataset)': '#2A9D8F',
}

In [ ]:
def parse_lr_label(model_id: str) -> str:
    match = re.search(r'-lr(?P<mantissa>\d+)e(?P<exp>\d+)(?:_\d+)?(?:-|$)', str(model_id))
    if match is None:
        return np.nan
    return f"{match.group('mantissa')}e-{match.group('exp')}"


def lr_label_to_float(lr_label: str) -> float:
    mantissa, exponent = lr_label.split('e-')
    return float(mantissa) * 10 ** (-int(exponent))


def add_benchmark_average(df: pd.DataFrame, group_cols: list[str], value_cols: list[str]) -> pd.DataFrame:
    df_avg = df.groupby(group_cols, dropna=False)[value_cols].mean().reset_index()
    df_avg['benchmark_name'] = 'benchmark_average'
    return pd.concat([df, df_avg], ignore_index=True)


def add_plot_labels(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df['benchmark_label'] = df['benchmark_name'].map(BENCHMARK_NAME_MAPPING)
    df['benchmark_order'] = df['benchmark_name'].map(BENCHMARK_ORDER)
    if 'finetuning_dataset' in df.columns:
        df['finetuning_dataset_label'] = df['finetuning_dataset'].map(FINETUNING_DATASET_LABELS)
    return df

In [ ]:
results_file = artifacts_dir / 'finetuning_results.csv'
df_all = pd.read_csv(results_file, low_memory=False)

df_all.shape

In [ ]:
def load_baseline(df_all: pd.DataFrame) -> pd.DataFrame:
    df_baseline = df_all[
        (df_all['exp_type'] == 'finetuning_baseline')
        & (df_all['model_id'] == BASE_MODEL_ID)
    ].copy()
    return apply_hiearchical_aggregation(
        df=df_baseline,
        groupby_cols=['model_id', 'benchmark_name'],
        agg_cols=AGG_COLS,
        agg_func='mean',
    )


def build_lora_diffs(df_all: pd.DataFrame, df_baseline: pd.DataFrame) -> pd.DataFrame:
    df_lora = df_all[
        (df_all['exp_type'] == 'finetuning_lora')
        & (df_all['data_pct'] == 100)
        & (df_all['arch'] == 'deit_small')
    ].copy()
    df_lora['lr_label'] = df_lora['model_id'].apply(parse_lr_label)
    df_lora['seed'] = df_lora['seed'].astype(int)
    df_lora['epochs'] = df_lora['epochs'].astype(int)
    df_lora['lora_rank'] = df_lora['lora_rank'].astype(int)
    df_lora['lora_alpha'] = df_lora['lora_alpha'].astype(int)
    df_lora['lora_config'] = df_lora.apply(lambda row: f"r={row['lora_rank']}, lr={row['lr_label']}", axis=1)

    df_lora = apply_hiearchical_aggregation(
        df=df_lora,
        groupby_cols=[
            'model_id',
            'benchmark_name',
            'seed',
            'finetuning_dataset',
            'epochs',
            'lr_label',
            'lora_rank',
            'lora_alpha',
            'lora_dropout',
            'lora_target_modules',
            'lora_update_head',
            'lora_config',
        ],
        agg_cols=AGG_COLS,
        agg_func='mean',
    )

    df_diff = df_lora.merge(df_baseline, on='benchmark_name', suffixes=('_finetuned', '_baseline'))
    for metric in AGG_COLS:
        df_diff[f'{metric}_diff'] = df_diff[f'{metric}_finetuned'] - df_diff[f'{metric}_baseline']

    group_cols = [
        'model_id_finetuned',
        'seed',
        'finetuning_dataset',
        'epochs',
        'lr_label',
        'lora_rank',
        'lora_alpha',
        'lora_dropout',
        'lora_target_modules',
        'lora_update_head',
        'lora_config',
    ]
    value_cols = [f'{metric}_diff' for metric in AGG_COLS]
    df_diff = add_benchmark_average(df_diff[group_cols + ['benchmark_name'] + value_cols], group_cols, value_cols)
    return add_plot_labels(df_diff)


def build_full_ft_diffs(df_all: pd.DataFrame, df_baseline: pd.DataFrame) -> pd.DataFrame:
    df_full = df_all[
        (df_all['exp_type'] == 'finetuning_explore')
        & (df_all['arch'] == 'deit_small')
        & (df_all['data_pct'] == 100)
    ].copy()
    df_full['lr_label'] = df_full['model_id'].apply(parse_lr_label)
    df_full['seed'] = df_full['seed'].astype(int)
    df_full['epochs'] = df_full['epochs'].astype(int)

    df_full = apply_hiearchical_aggregation(
        df=df_full,
        groupby_cols=['model_id', 'benchmark_name', 'seed', 'finetuning_dataset', 'epochs', 'lr_label'],
        agg_cols=AGG_COLS,
        agg_func='mean',
    )

    df_diff = df_full.merge(df_baseline, on='benchmark_name', suffixes=('_finetuned', '_baseline'))
    for metric in AGG_COLS:
        df_diff[f'{metric}_diff'] = df_diff[f'{metric}_finetuned'] - df_diff[f'{metric}_baseline']

    group_cols = ['model_id_finetuned', 'seed', 'finetuning_dataset', 'epochs', 'lr_label']
    value_cols = [f'{metric}_diff' for metric in AGG_COLS]
    df_diff = add_benchmark_average(df_diff[group_cols + ['benchmark_name'] + value_cols], group_cols, value_cols)
    return add_plot_labels(df_diff)


df_baseline = load_baseline(df_all)
df_lora = build_lora_diffs(df_all, df_baseline)
df_full = build_full_ft_diffs(df_all, df_baseline)

df_lora.shape, df_full.shape

## Experimental Coverage

LoRA runs cover all six fine-tuning datasets with ViT-S, $20$ epochs, one seed, ranks $\{8,16,32\}$, and learning rates $\{10^{-5},10^{-4},10^{-3}\}$. Full fine-tuning is evaluated at the paper setting of $20$ epochs and learning rate $10^{-5}$, with three seeds.

In [ ]:
coverage_summary = pd.DataFrame(
    {
        'experiment': ['LoRA', 'Full FT'],
        'finetuning_datasets': [df_lora['finetuning_dataset'].nunique(), df_full['finetuning_dataset'].nunique()],
        'evaluation_benchmarks': [df_lora['benchmark_name'].nunique() - 1, df_full['benchmark_name'].nunique() - 1],
        'learning_rates': [df_lora['lr_label'].nunique(), df_full['lr_label'].nunique()],
        'epochs': [df_lora['epochs'].nunique(), df_full['epochs'].nunique()],
        'seeds': [df_lora['seed'].nunique(), df_full['seed'].nunique()],
    }
)

coverage_summary

## LoRA Hyperparameters Across Fine-Tuning Datasets

Each LoRA setting is summarized by benchmark-average $\Delta r_{\mathrm{NC}}$ within each fine-tuning dataset and then averaged across datasets. The neural-transfer analyses use one shared overall LoRA setting, $20$ epochs, $r=32$, and learning rate $10^{-4}$. A separate descriptive panel reports dataset-specific selections.

In [ ]:
lora_overall_summary = (
    df_lora[df_lora['benchmark_name'] == 'benchmark_average']
    .groupby(['lora_config', 'lora_rank', 'lr_label'], dropna=False)
    .agg(
        pearsonr_nc_mean=('pearsonr_nc_diff', 'mean'),
        pearsonr_nc_median=('pearsonr_nc_diff', 'median'),
        rsa_c_test_mean=('rsa_c_test_diff', 'mean'),
        cka_c_test_mean=('cka_c_test_diff', 'mean'),
    )
    .reset_index()
)
lora_overall_summary['lr_float'] = lora_overall_summary['lr_label'].apply(lr_label_to_float)
lora_overall_summary = lora_overall_summary.sort_values(
    ['pearsonr_nc_mean', 'pearsonr_nc_median', 'lora_rank'],
    ascending=[False, False, False],
).reset_index(drop=True)

lora_per_dataset_summary = (
    df_lora[df_lora['benchmark_name'] == 'benchmark_average']
    .groupby(['finetuning_dataset', 'finetuning_dataset_label', 'lora_config', 'lora_rank', 'lr_label'], dropna=False)
    .agg(
        pearsonr_nc_mean=('pearsonr_nc_diff', 'mean'),
        rsa_c_test_mean=('rsa_c_test_diff', 'mean'),
        cka_c_test_mean=('cka_c_test_diff', 'mean'),
    )
    .reset_index()
)

best_lora_config = lora_overall_summary.iloc[0]['lora_config']

best_by_dataset = (
    lora_per_dataset_summary
    .sort_values(['finetuning_dataset', 'pearsonr_nc_mean'], ascending=[True, False])
    .groupby('finetuning_dataset')
    .head(1)
    .sort_values('pearsonr_nc_mean', ascending=False)
)

display(lora_overall_summary[['lora_config', 'pearsonr_nc_mean', 'pearsonr_nc_median', 'rsa_c_test_mean', 'cka_c_test_mean']])
display(best_by_dataset[['finetuning_dataset_label', 'lora_config', 'pearsonr_nc_mean']])

In [ ]:
lora_heatmap_df = lora_per_dataset_summary.copy()
lr_order = sorted(lora_heatmap_df['lr_label'].dropna().unique(), key=lr_label_to_float)
rank_order = sorted(lora_heatmap_df['lora_rank'].dropna().unique())
heatmap_absmax = np.nanmax(np.abs(lora_heatmap_df['pearsonr_nc_mean'] * HEATMAP_VAL_MULTIPLIER))

fig, axes = plt.subplots(2, 3, figsize=(12.0, 6.6), dpi=300)
axes = axes.flatten()

for ax, dataset in zip(axes, FINETUNING_DATASETS):
    data = lora_heatmap_df[lora_heatmap_df['finetuning_dataset'] == dataset].copy()
    heatmap_data = data.pivot_table(index='lora_rank', columns='lr_label', values='pearsonr_nc_mean', aggfunc='mean').loc[rank_order, lr_order] * HEATMAP_VAL_MULTIPLIER
    sns.heatmap(
        heatmap_data,
        annot=True,
        fmt='.2f',
        cmap='vlag',
        center=0,
        vmin=-heatmap_absmax,
        vmax=heatmap_absmax,
        linewidths=0.5,
        linecolor='white',
        cbar=False,
        ax=ax,
    )
    ax.set_title(FINETUNING_DATASET_LABELS[dataset], fontsize=14, fontweight='bold')
    ax.set_xlabel('Learning Rate', fontweight='bold')
    ax.set_ylabel('LoRA Rank', fontweight='bold')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0)

fig.suptitle('LoRA Hyperparameter Sweep by Fine-Tuning Dataset', fontsize=18, fontweight='bold')
fig.tight_layout(rect=[0, 0, 0.9, 1])
cbar_ax = fig.add_axes([0.92, 0.15, 0.015, 0.7])
fig.colorbar(axes[-1].collections[0], cax=cbar_ax, label=r'$100 \times \Delta r_{\mathrm{NC}}$ (benchmark average)')

save_figs(
    fig=fig,
    save_dir=save_dir_supp,
    base_filename='finetuning-lora-hyperparams-by_finetuning_dataset_heatmap',
    dpi=300,
    formats=['pdf', 'png', 'svg'],
)

**Caption: LoRA Hyperparameter Sweep by Fine-Tuning Dataset**

\textbf{finetuning-lora-hyperparams-by_finetuning_dataset_heatmap:} Benchmark-average $100 \times \Delta r_{\mathrm{NC}}$ for each LoRA rank and learning rate, shown separately for each fine-tuning dataset. Learning rate has a stronger influence than rank, with $10^{-4}$ performing best for most datasets.

In [ ]:
lora_by_benchmark_summary = (
    df_lora
    .groupby(['benchmark_name', 'benchmark_label', 'lora_rank', 'lr_label'], dropna=False)
    .agg(pearsonr_nc_mean=('pearsonr_nc_diff', 'mean'))
    .reset_index()
)

lr_order = sorted(lora_by_benchmark_summary['lr_label'].dropna().unique(), key=lr_label_to_float)
rank_order = sorted(lora_by_benchmark_summary['lora_rank'].dropna().unique())
benchmark_heatmap_values = lora_by_benchmark_summary['pearsonr_nc_mean'] * HEATMAP_VAL_MULTIPLIER
heatmap_absmax = np.nanmax(np.abs(benchmark_heatmap_values))

fig, axes = plt.subplots(3, 3, figsize=(12.0, 9.0), dpi=300)
axes = axes.flatten()

for idx, (ax, benchmark) in enumerate(zip(axes, BENCHMARKS_WITH_AVERAGE)):
    data = lora_by_benchmark_summary[lora_by_benchmark_summary['benchmark_name'] == benchmark].copy()
    heatmap_data = data.pivot_table(index='lora_rank', columns='lr_label', values='pearsonr_nc_mean', aggfunc='mean').loc[rank_order, lr_order] * HEATMAP_VAL_MULTIPLIER
    sns.heatmap(
        heatmap_data,
        annot=True,
        fmt='.2f',
        cmap='vlag',
        center=0,
        vmin=-heatmap_absmax,
        vmax=heatmap_absmax,
        linewidths=0.5,
        linecolor='white',
        cbar=False,
        ax=ax,
    )
    ax.set_title(BENCHMARK_NAME_MAPPING[benchmark], fontsize=14, fontweight='bold')
    ax.set_xlabel('Learning Rate', fontweight='bold')
    ax.set_ylabel('LoRA Rank' if idx % 3 == 0 else '', fontweight='bold')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0)

fig.suptitle('LoRA Hyperparameter Sweep by Evaluation Benchmark', fontsize=18, fontweight='bold')
fig.tight_layout(rect=[0, 0, 0.9, 1])
cbar_ax = fig.add_axes([0.92, 0.15, 0.015, 0.7])
fig.colorbar(axes[-1].collections[0], cax=cbar_ax, label=r'$100 \times \Delta r_{\mathrm{NC}}$')

save_figs(
    fig=fig,
    save_dir=save_dir_supp,
    base_filename='finetuning-lora-hyperparams-by_benchmark_heatmap',
    dpi=300,
    formats=['pdf', 'png', 'svg'],
)

**Caption: LoRA Hyperparameter Sweep by Evaluation Benchmark**

\textbf{finetuning-lora-hyperparams-by_benchmark_heatmap:} LoRA hyperparameter effects, $100 \times \Delta r_{\mathrm{NC}}$, separated by evaluation benchmark and averaged across fine-tuning datasets. The shared $10^{-4}$ setting gives the strongest average transfer, whereas $10^{-3}$ generally reduces alignment.

In [ ]:
lora_same_dataset_summary = (
    df_lora[df_lora['finetuning_dataset'] == df_lora['benchmark_name']]
    .groupby(['finetuning_dataset', 'finetuning_dataset_label', 'lora_rank', 'lr_label'], dropna=False)
    .agg(pearsonr_nc_mean=('pearsonr_nc_diff', 'mean'))
    .reset_index()
)

lr_order = sorted(lora_same_dataset_summary['lr_label'].dropna().unique(), key=lr_label_to_float)
rank_order = sorted(lora_same_dataset_summary['lora_rank'].dropna().unique())
same_dataset_heatmap_values = lora_same_dataset_summary['pearsonr_nc_mean'] * HEATMAP_VAL_MULTIPLIER
heatmap_absmax = np.nanmax(np.abs(same_dataset_heatmap_values))

fig, axes = plt.subplots(2, 3, figsize=(12.0, 6.6), dpi=300)
axes = axes.flatten()

for idx, (ax, dataset) in enumerate(zip(axes, FINETUNING_DATASETS)):
    data = lora_same_dataset_summary[lora_same_dataset_summary['finetuning_dataset'] == dataset].copy()
    heatmap_data = data.pivot_table(index='lora_rank', columns='lr_label', values='pearsonr_nc_mean', aggfunc='mean').loc[rank_order, lr_order] * HEATMAP_VAL_MULTIPLIER
    sns.heatmap(
        heatmap_data,
        annot=True,
        fmt='.2f',
        cmap='vlag',
        center=0,
        vmin=-heatmap_absmax,
        vmax=heatmap_absmax,
        linewidths=0.5,
        linecolor='white',
        cbar=False,
        ax=ax,
    )
    ax.set_title(FINETUNING_DATASET_LABELS[dataset], fontsize=14, fontweight='bold')
    ax.set_xlabel('Learning Rate', fontweight='bold')
    ax.set_ylabel('LoRA Rank' if idx % 3 == 0 else '', fontweight='bold')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0)

fig.suptitle('LoRA Hyperparameter Sweep: Same Fine-Tuning and Evaluation Dataset', fontsize=18, fontweight='bold')
fig.tight_layout(rect=[0, 0, 0.9, 1])
cbar_ax = fig.add_axes([0.92, 0.15, 0.015, 0.7])
fig.colorbar(axes[-1].collections[0], cax=cbar_ax, label=r'$100 \times \Delta r_{\mathrm{NC}}$')

save_figs(
    fig=fig,
    save_dir=save_dir_supp,
    base_filename='finetuning-lora-hyperparams-same_dataset_heatmap',
    dpi=300,
    formats=['pdf', 'png', 'svg'],
)

**Caption: LoRA Hyperparameter Sweep for Matching Fine-Tuning and Evaluation Datasets**

\textbf{finetuning-lora-hyperparams-same_dataset_heatmap:} LoRA hyperparameter effects, $100 \times \Delta r_{\mathrm{NC}}$, when the fine-tuning and evaluation dataset match. Matching-dataset gains remain sensitive to learning rate and do not support one dataset-specific optimum as a general evaluation setting.

## Overall LoRA vs Full Fine-Tuning

The shared overall LoRA configuration is trained for $20$ epochs with $r=32$ and learning rate $10^{-4}$, which has the largest benchmark-average $\Delta r_{\mathrm{NC}}$ when the tested LoRA settings are averaged across fine-tuning datasets. It is used consistently in all comparisons below except the explicitly labeled dataset-specific sensitivity panel. Full fine-tuning uses the paper setting of $20$ epochs and learning rate $10^{-5}$ throughout.

In [ ]:
full_ft_summary = (
    df_full[df_full['benchmark_name'] == 'benchmark_average']
    .groupby(['epochs', 'lr_label'])
    .agg(
        pearsonr_nc_mean=('pearsonr_nc_diff', 'mean'),
        pearsonr_nc_std=('pearsonr_nc_diff', 'std'),
        rsa_c_test_mean=('rsa_c_test_diff', 'mean'),
        cka_c_test_mean=('cka_c_test_diff', 'mean'),
    )
    .reset_index()
    .sort_values('pearsonr_nc_mean', ascending=False)
    .reset_index(drop=True)
)

best_full_epochs = FULL_FT_EPOCHS
best_full_lr = FULL_FT_LR_LABEL

best_lora_df = df_lora[df_lora['lora_config'] == best_lora_config].copy()
best_full_df = df_full[(df_full['epochs'] == best_full_epochs) & (df_full['lr_label'] == best_full_lr)].copy()

best_lora_avg = (
    best_lora_df
    .groupby(['finetuning_dataset', 'benchmark_name', 'benchmark_label', 'benchmark_order'], dropna=False)
    .agg(
        pearsonr_nc_diff=('pearsonr_nc_diff', 'mean'),
        rsa_c_test_diff=('rsa_c_test_diff', 'mean'),
        cka_c_test_diff=('cka_c_test_diff', 'mean'),
    )
    .reset_index()
)

best_full_avg = (
    best_full_df
    .groupby(['finetuning_dataset', 'benchmark_name', 'benchmark_label', 'benchmark_order'], dropna=False)
    .agg(
        pearsonr_nc_diff=('pearsonr_nc_diff', 'mean'),
        pearsonr_nc_diff_std=('pearsonr_nc_diff', 'std'),
        rsa_c_test_diff=('rsa_c_test_diff', 'mean'),
        cka_c_test_diff=('cka_c_test_diff', 'mean'),
    )
    .reset_index()
)

comparison_df = best_lora_avg.merge(
    best_full_avg,
    on=['finetuning_dataset', 'benchmark_name', 'benchmark_label', 'benchmark_order'],
    suffixes=('_lora', '_full'),
)
comparison_df['pearsonr_nc_gap'] = comparison_df['pearsonr_nc_diff_lora'] - comparison_df['pearsonr_nc_diff_full']
comparison_df['rsa_c_test_gap'] = comparison_df['rsa_c_test_diff_lora'] - comparison_df['rsa_c_test_diff_full']
comparison_df['cka_c_test_gap'] = comparison_df['cka_c_test_diff_lora'] - comparison_df['cka_c_test_diff_full']

summary_md = (
    f"**Overall LoRA setting:** `{best_lora_config}`  \n"
    f"Mean benchmark-average $\\Delta r_{{\\mathrm{{NC}}}}$ across fine-tuning datasets: `{lora_overall_summary.iloc[0]['pearsonr_nc_mean']:.4f}`\n\n"
    f"**Full-FT setting:** `ep{best_full_epochs}/lr{best_full_lr}`  \n"
    f"Mean benchmark-average $\\Delta r_{{\\mathrm{{NC}}}}$ across fine-tuning datasets and seeds: `{best_full_df[best_full_df['benchmark_name'] == 'benchmark_average']['pearsonr_nc_diff'].mean():.4f}`"
)
display(Markdown(summary_md))
display(full_ft_summary)

In [ ]:
dataset_specific_best_lora = (
    df_lora[df_lora['benchmark_name'] == 'benchmark_average']
    .merge(
        best_by_dataset[['finetuning_dataset', 'lora_config']],
        on=['finetuning_dataset', 'lora_config'],
        how='inner',
    )
    .groupby(['finetuning_dataset', 'finetuning_dataset_label', 'lora_config'], dropna=False)
    .agg(pearsonr_nc_diff=('pearsonr_nc_diff', 'mean'))
    .reset_index()
)
dataset_specific_best_lora['plot_label'] = dataset_specific_best_lora.apply(
    lambda row: f"{row['finetuning_dataset_label']}\n{row['lora_config']}", axis=1,
)
dataset_specific_best_lora['method'] = 'Best LoRA (per dataset)'

dataset_specific_full_reference = (
    best_full_avg[best_full_avg['benchmark_name'] == 'benchmark_average']
    [['finetuning_dataset', 'pearsonr_nc_diff']]
    .merge(dataset_specific_best_lora[['finetuning_dataset', 'plot_label']], on='finetuning_dataset')
)
dataset_specific_full_reference['method'] = 'Full FT'

dataset_specific_plot_df = pd.concat(
    [
        dataset_specific_full_reference[['finetuning_dataset', 'plot_label', 'method', 'pearsonr_nc_diff']],
        dataset_specific_best_lora[['finetuning_dataset', 'plot_label', 'method', 'pearsonr_nc_diff']],
    ],
    ignore_index=True,
)
dataset_specific_order = [
    dataset_specific_best_lora.set_index('finetuning_dataset').loc[dataset, 'plot_label']
    for dataset in FINETUNING_DATASETS
]

fig, ax = plt.subplots(figsize=(8.2, 4.4), dpi=300)
sns.barplot(
    data=dataset_specific_plot_df,
    x='plot_label',
    y='pearsonr_nc_diff',
    order=dataset_specific_order,
    hue='method',
    hue_order=['Full FT', 'Best LoRA (per dataset)'],
    palette={'Full FT': METHOD_PALETTE['Full FT'], 'Best LoRA (per dataset)': METHOD_PALETTE['Best LoRA (per dataset)']},
    ax=ax,
)
ax.axhline(0, color='0.2', linewidth=0.8)
ax.set_title('Dataset-Specific LoRA Sensitivity vs Full Fine-Tuning', fontsize=14, fontweight='bold')
ax.set_xlabel('Fine-Tuning Dataset and Selected LoRA Configuration', fontweight='bold')
ax.set_ylabel(r'$\Delta r_{\mathrm{NC}}$ (Benchmark Average)', fontweight='bold')
ax.tick_params(axis='x', labelsize=8.5)
ax.grid(axis='y', linestyle='--', alpha=0.35)
ax.spines[['top', 'right']].set_visible(False)
ax.legend(title=None, frameon=False, loc='upper left')
fig.tight_layout()

save_figs(
    fig=fig,
    save_dir=save_dir_supp,
    base_filename='finetuning-lora-dataset_specific_best-vs_full-benchmark_average',
    dpi=300,
    formats=['pdf', 'png', 'svg'],
)


**Caption: Dataset-Specific LoRA Sensitivity Comparison**

\textbf{finetuning-lora-dataset_specific_best-vs_full-benchmark_average:} Benchmark-average $\Delta r_{\mathrm{NC}}$ for post-hoc dataset-specific LoRA selections and fixed full fine-tuning at $20$ epochs and learning rate $10^{-5}$. Dataset-specific selection raises LoRA's descriptive upper bound but is not used for the primary comparison.

In [ ]:
gap_heatmap_df = comparison_df[comparison_df['benchmark_name'].isin(BENCHMARKS)][['finetuning_dataset', 'benchmark_name', 'pearsonr_nc_gap']].copy()
row_avg = gap_heatmap_df.groupby('finetuning_dataset', dropna=False)['pearsonr_nc_gap'].mean().reset_index()
row_avg['benchmark_name'] = 'benchmark_average'
col_avg = gap_heatmap_df.groupby('benchmark_name', dropna=False)['pearsonr_nc_gap'].mean().reset_index()
col_avg['finetuning_dataset'] = 'dataset_average'
corner = pd.DataFrame({'finetuning_dataset': ['dataset_average'], 'benchmark_name': ['benchmark_average'], 'pearsonr_nc_gap': [gap_heatmap_df['pearsonr_nc_gap'].mean()]})
heatmap_aug = pd.concat([gap_heatmap_df, row_avg, col_avg, corner], ignore_index=True)

row_order = FINETUNING_DATASETS + ['dataset_average']
col_order = BENCHMARKS + ['benchmark_average']
heatmap_matrix = heatmap_aug.pivot_table(index='finetuning_dataset', columns='benchmark_name', values='pearsonr_nc_gap').loc[row_order, col_order] * HEATMAP_VAL_MULTIPLIER
heatmap_matrix.index = [FINETUNING_DATASET_LABELS.get(idx, 'Average across datasets') for idx in heatmap_matrix.index]
heatmap_matrix.columns = [BENCHMARK_NAME_MAPPING.get(col, 'Average') for col in heatmap_matrix.columns]
heatmap_absmax = np.nanmax(np.abs(heatmap_matrix.values))

fig, ax = plt.subplots(figsize=(9.6, 4.8), dpi=300)
sns.heatmap(
    heatmap_matrix,
    annot=True,
    fmt='.2f',
    cmap='vlag',
    center=0,
    vmin=-heatmap_absmax,
    vmax=heatmap_absmax,
    linewidths=0.5,
    linecolor='white',
    cbar_kws={'label': r'Overall LoRA minus Full FT in $100 \times \Delta r_{\mathrm{NC}}$'},
    ax=ax,
)
ax.set_title('Overall LoRA Minus Full Fine-Tuning', fontsize=14, fontweight='bold')
ax.set_xlabel('Evaluation Benchmark')
ax.set_ylabel('Fine-Tuning Dataset')
ax.tick_params(axis='x', rotation=35)
ax.tick_params(axis='y', rotation=0)
fig.tight_layout()

save_figs(
    fig=fig,
    save_dir=save_dir_supp,
    base_filename='finetuning-lora-overall_vs_full-gap_heatmap',
    dpi=300,
    formats=['pdf', 'png', 'svg'],
)

**Caption: Overall LoRA Minus Full Fine-Tuning**

\textbf{finetuning-lora-overall_vs_full-gap_heatmap:} Difference in $100 \times \Delta r_{\mathrm{NC}}$ between overall LoRA ($20$ epochs, $r=32$, learning rate $10^{-4}$) and full fine-tuning ($20$ epochs, learning rate $10^{-5}$) across fine-tuning datasets and evaluation benchmarks. Full fine-tuning has the larger mean gain, driven primarily by the T-EEG1 fine-tuning condition.

In [ ]:
scatter_df = comparison_df[comparison_df['benchmark_name'] != 'benchmark_average'].copy()
scatter_df['finetuning_dataset_label'] = scatter_df['finetuning_dataset'].map(FINETUNING_DATASET_LABELS)
scatter_df['benchmark_modality'] = scatter_df['benchmark_name'].map(MODALITY_MAP)
scatter_df['finetuning_modality'] = scatter_df['finetuning_dataset'].map(MODALITY_MAP)
scatter_df['transfer_type'] = np.select(
    [
        scatter_df['benchmark_name'] == scatter_df['finetuning_dataset'],
        scatter_df['benchmark_modality'] == scatter_df['finetuning_modality'],
    ],
    ['Same dataset', 'Same modality'],
    default='Cross modality',
)

transfer_type_order = ['Same dataset', 'Same modality', 'Cross modality']
lims = [
    min(scatter_df['pearsonr_nc_diff_full'].min(), scatter_df['pearsonr_nc_diff_lora'].min()),
    max(scatter_df['pearsonr_nc_diff_full'].max(), scatter_df['pearsonr_nc_diff_lora'].max()),
]
pad = 0.0015
lims = [lims[0] - pad, lims[1] + pad]

In [ ]:
zoom = 0.75
figsize = (8, 6)
figsize = (figsize[0] * zoom, figsize[1] * zoom)

# figsize

In [ ]:
finetuning_dataset_order = [FINETUNING_DATASET_LABELS[dataset] for dataset in FINETUNING_DATASETS]
dataset_marker_map = {
    'TVSD-EP': 'o',
    'T-fMRI': 's',
    'NSD-fMRI': 'D',
    'T-EEG1': '^',
    'T-EEG2': 'v',
    'T-MEG': 'P',
}
transfer_palette = {
    'Same dataset': '#F28E2B',
    'Same modality': '#B07CC6',
    'Cross modality': '#6B3F2A',
}

fig, ax = plt.subplots(figsize=figsize, dpi=300)
sns.scatterplot(
    data=scatter_df,
    x='pearsonr_nc_diff_full',
    y='pearsonr_nc_diff_lora',
    hue='transfer_type',
    hue_order=transfer_type_order,
    palette=transfer_palette,
    style='finetuning_dataset_label',
    style_order=finetuning_dataset_order,
    markers=dataset_marker_map,
    s=60,
    alpha=0.9,
    edgecolor='0.2',
    linewidth=0.45,
    legend=False,
    ax=ax,
)
ax.plot(lims, lims, linestyle='--', color='0.3', linewidth=1)
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.set_aspect('equal')
ax.set_title(r'LoRA vs. Full Fine-Tuning', fontsize=16, fontweight='bold')
ax.set_xlabel(r'Full Fine-Tuning $\Delta r_{\mathrm{NC}}$', fontsize=14, fontweight='bold')
ax.set_ylabel(r'LoRA $\Delta r_{\mathrm{NC}}$', fontsize=14, fontweight='bold')

transfer_handles = [
    plt.Line2D(
        [0], [0], marker='o', linestyle='', color='0.2',
        markerfacecolor=transfer_palette[transfer_type], markeredgecolor='0.2',
        markersize=6, label=transfer_type,
    )
    for transfer_type in transfer_type_order
]
dataset_handles = [
    plt.Line2D(
        [0], [0], marker=dataset_marker_map[dataset_label], linestyle='', color='0.2',
        markerfacecolor='0.65', markeredgecolor='0.2', markersize=6, label=dataset_label,
    )
    for dataset_label in finetuning_dataset_order
]
transfer_legend = ax.legend(
    handles=transfer_handles, title='Transfer type', loc='upper left',
    frameon=True, framealpha=0.9, fontsize=10, title_fontsize=14,
)
ax.add_artist(transfer_legend)
ax.legend(
    handles=dataset_handles, title='Fine-tuning dataset', loc='lower left',
    frameon=True, framealpha=0.9, fontsize=10, title_fontsize=14, ncol=2,
    columnspacing=0.8, handletextpad=0.35,
)
ax.spines[['top', 'right']].set_visible(False)
ax.grid(True, linestyle='--', alpha=0.25)
fig.tight_layout()

save_figs(
    fig=fig,
    save_dir=save_dir_main,
    base_filename='finetuning-lora-overall_vs_full-neural_transfer',
    dpi=300,
    formats=['pdf', 'png', 'svg'],
)

**Caption: LoRA vs Full Fine-Tuning Across Neural Transfers**

\textbf{finetuning-lora-overall_vs_full-neural_transfer:} Neural alignment gain for overall LoRA ($20$ epochs, $r=32$, learning rate $10^{-4}$) versus full fine-tuning ($20$ epochs, learning rate $10^{-5}$) across neural transfer pairs. Colors denote transfer relationship and markers denote the fine-tuning dataset; overall LoRA remains competitive in many pairs, while full fine-tuning is stronger on average.

In [ ]:
neural_transfer_comparison_df = comparison_df[comparison_df['benchmark_name'].isin(BENCHMARKS)].copy()

win_summary = pd.DataFrame(
    {
        'metric': ['pearsonr_nc', 'rsa_c_test', 'cka_c_test'],
        'wins': [
            int((neural_transfer_comparison_df['pearsonr_nc_gap'] > 0).sum()),
            int((neural_transfer_comparison_df['rsa_c_test_gap'] > 0).sum()),
            int((neural_transfer_comparison_df['cka_c_test_gap'] > 0).sum()),
        ],
        'losses': [
            int((neural_transfer_comparison_df['pearsonr_nc_gap'] < 0).sum()),
            int((neural_transfer_comparison_df['rsa_c_test_gap'] < 0).sum()),
            int((neural_transfer_comparison_df['cka_c_test_gap'] < 0).sum()),
        ],
    }
)
win_summary['total_cells'] = win_summary['wins'] + win_summary['losses']
win_summary['win_rate'] = win_summary['wins'] / win_summary['total_cells']

overall_gap_summary = pd.DataFrame(
    {
        'metric': ['pearsonr_nc', 'rsa_c_test', 'cka_c_test'],
        'mean_gap_lora_minus_full': [
            neural_transfer_comparison_df['pearsonr_nc_gap'].mean(),
            neural_transfer_comparison_df['rsa_c_test_gap'].mean(),
            neural_transfer_comparison_df['cka_c_test_gap'].mean(),
        ],
    }
)

display(win_summary)
display(overall_gap_summary)

## Overall Analysis

### Main Figure: Neural Transfer

\paragraph{LoRA and full fine-tuning across neural transfers.} We implement LoRA by adding trainable low-rank updates to the attention and feed-forward transformations of ViT-S while allowing the classification head to adapt; the remaining vision-transformer weights remain fixed. Full implementation details and hyperparameter sensitivity analyses are provided in the appendix. Figure~\ref{fig:finetuning-lora-overall_vs_full-neural_transfer} compares a single overall LoRA setting trained for $20$ epochs ($r=32$, learning rate $10^{-4}$) with full fine-tuning at $20$ epochs and learning rate $10^{-5}$ across all $48$ fine-tuning-dataset by evaluation-benchmark transfer pairs. Full fine-tuning yields the larger benchmark-average neural-alignment gain ($\Delta r_{\mathrm{NC}}=0.00591$) than overall LoRA ($0.00421$), a mean difference of $0.00169$. Overall LoRA exceeds full fine-tuning in $21$ of $48$ transfer pairs and remains close to full fine-tuning for several same-dataset and cross-dataset transfers, but its large loss after fine-tuning on T-EEG1 reduces its average performance. Thus, a shared LoRA configuration preserves much of the alignment benefit of full fine-tuning, while full fine-tuning produces the stronger average gain under the fixed comparison used in the main figure.

### Appendix: LoRA Hyperparameters and Comparison With Full Fine-Tuning

\paragraph{LoRA implementation.} We apply low-rank updates to the attention projection and both feed-forward transformations within each transformer block, while the classification head remains trainable. All LoRA runs use a scaling factor of $32$ and adapter dropout of $0.1$, introduce no adapted bias terms, and exclude the neural readout modules from low-rank adaptation. The sweep varies adapter rank over $\{8,16,32\}$ and learning rate over $\{10^{-5},10^{-4},10^{-3}\}$ at $20$ epochs.

\paragraph{LoRA hyperparameter sensitivity.} Figure~\ref{fig:finetuning-lora-hyperparams-by_finetuning_dataset_heatmap} shows that the LoRA learning rate has a larger effect than rank across fine-tuning datasets. Learning rate $10^{-4}$ gives the highest benchmark-average gain for four of six datasets, while NSD-fMRI favors $10^{-3}$ and T-EEG1 favors $10^{-5}$. When averaged across datasets, the largest gain occurs at $r=32$ and learning rate $10^{-4}$ ($\Delta r_{\mathrm{NC}}=0.00421$), closely followed by the other ranks at the same learning rate ($0.00417$ for $r=16$ and $0.00408$ for $r=8$). Figure~\ref{fig:finetuning-lora-hyperparams-by_benchmark_heatmap} shows the same learning-rate dependence across evaluation benchmarks: $10^{-4}$ generally supports positive transfer, whereas $10^{-3}$ substantially reduces average neural alignment. Figure~\ref{fig:finetuning-lora-hyperparams-same_dataset_heatmap} further shows that the optimal setting is not uniform when fine-tuning and evaluation use the same dataset, motivating the use of a shared overall LoRA setting for the main comparison rather than selecting a configuration independently for each reported outcome.

\paragraph{Dataset-specific sensitivity analysis.} Figure~\ref{fig:finetuning-lora-dataset_specific_best-vs_full-benchmark_average} reports the post-hoc upper-bound comparison in which each fine-tuning dataset uses its highest-scoring LoRA setting. This descriptive analysis increases the mean LoRA gain to $\Delta r_{\mathrm{NC}}=0.00680$, compared with $0.00591$ for fixed full fine-tuning, but it uses the reported evaluation outcomes for selection and therefore does not define the main comparison.

\paragraph{Shared LoRA versus full fine-tuning.} Figure~\ref{fig:finetuning-lora-overall_vs_full-gap_heatmap} compares the overall LoRA setting with full fine-tuning for each fine-tuning dataset and evaluation benchmark. Overall LoRA produces larger benchmark-average gains after fine-tuning on TVSD-EP, T-fMRI, NSD-fMRI, and T-MEG, is slightly lower for T-EEG2, and is substantially lower for T-EEG1 ($\Delta r_{\mathrm{NC}}$ difference $=-0.01155$). Consequently, the shared LoRA setting is competitive in several transfer regimes, but full fine-tuning has the larger average gain across datasets and benchmarks.